# Pre-entrega 6: Orquestador Multi-Agente - Presupuesto de PC Gaming

Demostración de un sistema multi-agente con **topología jerárquica** en LangGraph para cotizar y validar el armado de una PC según un presupuesto y consumo energético.

### Agentes y Nodos:
1. **Supervisor**: Router central que evalúa el estado y deriva tareas mediante salidas estructuradas (`Literal`).
2. **Investigador (`researcher`)**: Busca componentes de hardware, modelos, precios y consumo en Watts.
3. **Analista (`analyst`)**: Calcula el costo total contra el presupuesto y determina la potencia necesaria para la fuente de poder.
4. **Validador (`validator`)**: Evalúa que los datos y cálculos sean completos y redacta la recomendación final.

In [1]:
from dotenv import load_dotenv
from IPython.display import Image, display, Markdown
from langchain_core.messages import HumanMessage
from graph import build_graph, export_graph_mermaid

load_dotenv()
print("Entorno cargado.")

Entorno cargado.


## 1. Visualización del Grafo Jerárquico

In [2]:
app = build_graph()
mermaid_syntax = export_graph_mermaid(app)
Markdown(f"```mermaid\n{mermaid_syntax}\n```")

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	analyst(analyst)
	validator(validator)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analyst --> supervisor;
	researcher --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> analyst;
	supervisor -.-> researcher;
	supervisor -.-> validator;
	validator -.-> __end__;
	validator -.-> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc
```


## 2. Consulta y Estado Inicial

Probamos con una solicitud para armar una PC con un presupuesto tope de $850.000.

In [3]:
user_query = (
    "Tengo un presupuesto de $850.000 para armar una PC gamer. "
    "Recomiéndame procesador, placa de video, memoria RAM y fuente. "
    "Calcula si me alcanza el dinero y de cuántos Watts tiene que ser la fuente para que funcione bien."
)

initial_state = {
    "messages": [HumanMessage(content=user_query)],
    "user_query": user_query,
    "current_task": None,
    "research_data": None,
    "analysis_data": None,
    "agent_history": [],
    "next_agent": "supervisor",
    "supervisor_reasoning": None,
    "validation_passed": False,
    "validation_feedback": None,
    "final_synthesis": None,
    "iteration_count": 0,
    "max_iterations": 5
}

print("Consulta:", user_query)

Consulta: Tengo un presupuesto de $850.000 para armar una PC gamer. Recomiéndame procesador, placa de video, memoria RAM y fuente. Calcula si me alcanza el dinero y de cuántos Watts tiene que ser la fuente para que funcione bien.


## 3. Ejecución del Flujo con Delegación del Supervisor

In [4]:
config = {"recursion_limit": 15}
final_state = None

for output in app.stream(initial_state, config=config):
    for node_name, node_state in output.items():
        final_state = node_state
        print(f"\n--- Nodo: {node_name} ---")
        if node_name == "supervisor":
            print(f"Decisión: -> {node_state.get('next_agent')}")
            print(f"Motivo: {node_state.get('supervisor_reasoning')}")
            print(f"Tarea: {node_state.get('current_task')}")
        elif node_name == "researcher":
            print("Componentes encontrados por el investigador:")
            print(node_state.get('research_data')[:250] + "...")
        elif node_name == "analyst":
            print("Cálculos realizados por el analista:")
            print(node_state.get('analysis_data')[:250] + "...")
        elif node_name == "validator":
            print(f"Validación aprobada: {node_state.get('validation_passed')}")


--- Nodo: supervisor ---
Decisión: -> researcher
Motivo: Aún no se investigaron componentes para la consulta del usuario.
Tarea: Buscar procesador, placa de video, RAM y fuente según el presupuesto.

--- Nodo: researcher ---
Componentes encontrados por el investigador:
Componentes seleccionados para el armado:
[{"categoria": "procesador", "modelo": "AMD Ryzen 5 7600", "precio": 240000, "consumo_watts": 65}, ...]

--- Nodo: supervisor ---
Decisión: -> analyst
Motivo: Componentes recolectados. Se requiere calcular el costo total y el consumo energético.
Tarea: Calcular la suma total vs presupuesto y la potencia recomendada de la fuente.

--- Nodo: analyst ---
Cálculos realizados por el analista:
Análisis Financiero:
{"costo_total": 765000.0, "presupuesto_limite": 850000.0, "diferencia": 85000.0, "porcentaje_utilizado": 90.0}...

--- Nodo: supervisor ---
Decisión: -> validator
Motivo: Tanto la investigación como el análisis fueron completados. Procediendo a validar.
Tarea: Verificar la s

## 4. Presupuesto Final y Bitácora de Contribuciones

In [5]:
if final_state and final_state.get("final_synthesis"):
    display(Markdown(final_state["final_synthesis"]))

print("\n--- Historial de Contribuciones ---")
for item in final_state.get("agent_history", []):
    print(f"- [{item['agent']}]: {item['summary']}")

Presupuesto de PC Gamer Recomendado:

1. Procesador: AMD Ryzen 5 7600 ($240.000, 65W TDP, Socket AM5)
2. Placa de Video: NVIDIA GeForce RTX 4060 8GB ($360.000, 115W TDP, 1080p Ultra)
3. Memoria RAM: Corsair Vengeance 16GB DDR5 5600MHz ($85.000, 10W)
4. Fuente: Corsair CV650 650W 80 Plus Bronze ($80.000)

Balance Financiero:
- Costo Total: $765.000
- Presupuesto Límite: $850.000
- Margen Disponible: $85.000 (utilizaste el 90% del presupuesto)

Consumo Eléctrico:
- Consumo Estimado: ~230W
- Potencia Recomendada: Fuente de 550W a 650W 80 Plus (la fuente de 650W elegida trabaja con excelente margen térmico y eficiencia).



--- Historial de Contribuciones ---
- [supervisor]: Aún no se investigaron componentes para la consulta del usuario.
- [researcher]: Selección de componentes de PC con precios y consumo en Watts.
- [supervisor]: Componentes recolectados. Se requiere calcular el costo total y el consumo energético.
- [analyst]: Cálculo de presupuesto total, balance financiero y potencia de fuente requerida.
- [supervisor]: Tanto la investigación como el análisis fueron completados. Procediendo a validar.
- [validator]: Aprobado: True. Validación aprobada. Se cubrieron todos los componentes y cálculos solicitados.
